# FT-00a C# : LoRA from scratch — démonter l'adaptation bas-rang
**Navigation** : [Index](README.md) | [Suivant >>](FT-00b-LoRA-Hyperparams-from-scratch-CSharp.ipynb)

**Version anglaise** : [FT-00a — LoRA from scratch (en)](FT-00a-LoRA-from-scratch-Python_en.ipynb)

**Objectif** : comprendre LoRA en le construisant, sans `peft`. À la fin de ce notebook vous saurez écrire un `LoRALinear` avec TorchSharp, prouver que l'initialisation canonique ne change pas le modèle, l’entraîner sur une mini-tâche réelle et expliquer pourquoi la fusion de l’adaptateur peut changer les arrondis.

**Prérequis** : bases de TorchSharp (tenseurs, autograd, `nn.Linear`) et notions de fine-tuning — voir [FT-01](FT-01-Introduction-FineTuning-Python.ipynb).

**Durée** : ~30 min · **Niveau** : intermédiaire · **Matériel** : CPU, TorchSharp-cpu 0.107.0 et les archives Fashion-MNIST indiquées par `COURSIA_FASHION_MNIST`.

**Position dans la série** : FT-01 à FT-06 utilisent `peft` et des modèles HuggingFace prêts à l'emploi. Ce notebook démonte le mécanisme — la décomposition `W' = W + (alpha/r) * B @ A`, l'initialisation `A ~ N, B = 0`, la propagation du gradient, la fusion — pour que les notebooks SOTA cessent d'être des boîtes noires. Les notebooks suivants restent la référence pour l'usage industriel : [FT-01](FT-01-Introduction-FineTuning-Python.ipynb) (LoRA avec `peft` sur GPT-2), [FT-02](FT-02-QLoRA-Quantization-Python.ipynb) (QLoRA 4-bit), [FT-06](FT-06-Vision-Language-LoRA-Python.ipynb) (vision-langage).

[Version Python](FT-00a-LoRA-from-scratch-Python.ipynb)

### Vérification de l’environnement

La cellule affiche la version de TorchSharp, celle de libtorch, le CPU, le nombre de threads et la graine de ce replay.

In [1]:
#r "nuget:TorchSharp-cpu, 0.107.0"
// Configurer le moteur CPU et la graine pour le banc complet.
using System;
using System.IO;
using System.IO.Compression;
using System.Buffers.Binary;
using System.Diagnostics;
using System.Linq;
using System.Collections.Generic;
using TorchSharp;
using TorchSharp.Modules;
using static TorchSharp.torch;
using Tensor = TorchSharp.torch.Tensor;

torch.set_num_threads(2);
torch.manual_seed(42);
Console.WriteLine($"TorchSharp {typeof(torch).Assembly.GetName().Version}, libtorch {torch.__version__}, CPU, threads {torch.get_num_threads()}, graine 42");


Installed Packages TorchSharp-cpu, 0.107.0

Loading extensions from `/workspace/coursia-toolchains/nuget/skiasharp/2.88.6/interactive-extensions/dotnet/SkiaSharp.DotNet.Interactive.dll`

TorchSharp 0.107.0.0, libtorch 2.10.0.0, CPU, threads 2, graine 42


### Moteur effectif

La cellule affiche les versions, le CPU et le nombre de threads de ce replay. Les temps dépendent de la machine.

## 1. Le problème : `W' = W + delta_W` coûte `|delta_W|` paramètres

Le **full fine-tuning** apprend un delta complet : pour une couche `d -> k`, il faut stocker et mettre à jour `d x k` paramètres — plus les états d'optimiseur (Adam en porte deux copies : moment et variance). Sur GPT-2 (124 M de paramètres), c'est le prix intégral, même quand la tâche cible ne déplace le modèle que dans un tout petit sous-espace.

L'hypothèse de LoRA ([Hu et al., 2021](https://arxiv.org/abs/2106.09685)) est que ce delta a un **rang intrinsèquement faible** :

$$W' = W + \frac{\alpha}{r} \cdot B A, \qquad A \in \mathbb{R}^{r \times d},\ B \in \mathbb{R}^{k \times r},\ r \ll \min(d, k)$$

Le budget passe de `d x k` à `r (d + k)`. Sur la couche jouet de ce notebook (`512 -> 10`, `r = 4`) :

| | full | LoRA r=4 |
|---|---|---|
| paramètres du delta | 512 x 10 = **5 120** | 4 x (512 + 10) = **2 088** |
| part du delta | 100 % | 40,8 % |

Deux choix de conception font que cette décomposition est **entraînable depuis le modèle de base, sans le déranger** :

1. **`B = 0` à l'initialisation** : le delta vaut exactement zéro, donc `W' = W` au pas 0 — le modèle adapté démarre *identique* au modèle de base, quel que soit `A`.
2. **`W` est gelé** (`requires_grad = False`) : les poids gelés ne reçoivent pas de gradient. Le gradient par rapport à l’entrée reste calculable quand un adaptateur en amont en a besoin ; un adaptateur entraîné se stocke en `r (d + k)` valeurs au lieu de `d x k`.

Le facteur `alpha / r` (le *scaling*) règle le facteur multiplicatif. Garder `alpha / r` constant en changeant `r` ne garantit pas une norme constante du delta appris : A et B changent aussi.

### `LoRALinear` : la décomposition avec TorchSharp

Pas d'import de `peft` ci-dessous — la classe tient en une vingtaine de lignes, et chacun de ses détails est l'un des trois invariants de la section 1.

In [2]:
// La base est gelée ; seules les matrices de bas rang reçoivent des gradients.
public class LoRALinear : torch.nn.Module<Tensor, Tensor>
{
    public readonly Linear Base;
    public readonly Parameter A, B;
    public readonly double Scaling;
    public LoRALinear(Linear b, int rank, double alpha) : base("LoRALinear")
    {
        if (rank <= 0)
            throw new ArgumentException("rang strictement positif");
        Base = b;
        foreach (var p in Base.parameters())
            p.requires_grad_(false);
        Scaling = alpha / rank;
        A = torch.nn.Parameter(torch.randn(new long[] { rank, b.weight.shape[1] }) / Math.Sqrt(b.weight.shape[1]));
        B = torch.nn.Parameter(torch.zeros(new long[] { b.weight.shape[0], rank }));
        RegisterComponents();
    }

    public override Tensor forward(Tensor x) => Base.forward(x) + Scaling * torch.matmul(torch.matmul(x, A.transpose(0, 1)), B.transpose(0, 1));
    public Tensor MergedWeight() => Base.weight + Scaling * torch.matmul(B, A);
}

Console.WriteLine("LoRALinear : base gelée, A aléatoire, B nul, scaling alpha/r.");


LoRALinear : base gelée, A aléatoire, B nul, scaling alpha/r.


### Le mécanisme

La couche de base est gelée. A est initialisée aléatoirement et B à zéro : le delta est nul au départ ; seuls A et B sont entraînables.

La base garde son poids dense et son biais : la couche additionne à sa sortie un terme calculé par deux produits matriciels de rang réduit. Si l’entrée a `d` composantes et la sortie `k`, les facteurs ont respectivement `r × d` et `k × r` éléments. Leur budget est donc `r(d+k)`, contre `dk` pour une matrice dense sans biais. Le facteur `alpha/r` multiplie le delta effectif ; il est distinct du nombre de paramètres.

À l’initialisation, A fournit des directions aléatoires et B vaut zéro. Le produit `B A` est alors nul, ce qui rend exactement la sortie de l’adaptateur égale à celle de la couche de base. Après le premier backward, A peut avoir un gradient numériquement nul parce qu’il est multiplié par B ; sa présence et sa valeur sont deux propriétés différentes. B, elle, peut recevoir un gradient dès cette première étape.

## 2. Les invariants, vérifiés sur tenseurs réels

Affirmer « le delta vaut zéro au départ » et « W ne reçoit pas de gradient » est une chose ; le vérifier sur le graphe autograd réel en est une autre. Les trois contrôles ci-dessous sont ceux qu'on écrirait dans un test unitaire du module.

In [3]:
// Vérifier identité initiale, gradients et budget avant tout entraînement.
var baseLayer = torch.nn.Linear(512, 10);
var layer = new LoRALinear(baseLayer, 4, 8);
var probe = torch.randn(new long[] { 64, 512 });
var initial = (layer.forward(probe) - baseLayer.forward(probe)).abs().max().item<float>();
var lossInvariant = layer.forward(probe).square().mean();
lossInvariant.backward();
var gradA = layer.A.grad.abs().max().item<float>();
var gradB = layer.B.grad.abs().max().item<float>();
Console.WriteLine($"norme max gradient A={gradA:E3}, B={gradB:E3}");
if (gradA != 0 || gradB <= 0)
    throw new Exception("gradients initiaux inattendus");
var nTrain = layer.parameters().Where(p => p.requires_grad).Sum(p => p.numel());
var total = layer.parameters().Sum(p => p.numel());
Console.WriteLine($"écart initial {initial:E3}; gradient W {(baseLayer.weight.grad is null ? "absent" : "présent")}; gradients A/B {layer.A.grad is not null}/{layer.B.grad is not null}");
Console.WriteLine($"paramètres entraînables {nTrain} = {4 * (512 + 10)} ; total {total}");
if (initial != 0 || baseLayer.weight.grad is not null || layer.A.grad is null || layer.B.grad is null || nTrain != 2088)
    throw new Exception("invariant LoRA violé");


norme max gradient A=0.000E+000, B=1.032E-001


écart initial 0.000E+000; gradient W absent; gradients A/B True/True


paramètres entraînables 2088 = 2088 ; total 7218


### Les invariants

Le test ci-dessus vérifie l’identité initiale, l’absence de gradient sur W, la présence de gradients sur A et B et le budget.

Le test choisit une couche `512 → 10`, un rang 4 et `alpha=8`. Le budget LoRA est `4 × (512 + 10) = 2 088` paramètres. Le poids dense et le biais de la base restent présents dans le module, mais ne participent pas à l’optimisation. Cela explique la différence entre nombre total et nombre entraînable affichés.

Le contrôle compare les deux sorties sur le même tenseur avant tout apprentissage. Il déclenche ensuite un backward sur une perte quadratique et vérifie séparément que le gradient du poids gelé est absent et que ceux des deux facteurs existent. Ces observations concernent l’état initial de cette couche ; elles ne prédisent pas l’exactitude de la tâche Fashion-MNIST.

### Exercice 1 — Le rang et le budget

Compléter `Budget` pour compter les paramètres appris de l’adaptateur linéaire 512 → 10. Comparer au poids dense de 5 120 coefficients pour les rangs 1, 2, 4, 8, 16 et 64.

**Indice.** Utiliser les dimensions de A et B vérifiées ci-dessus. À quel rang l’adaptateur devient-il plus coûteux que le poids dense ? Quel rang maximal un produit de matrice de taille 10 × 512 peut-il effectivement représenter ?


In [4]:
int? Budget(int rank, int d = 512, int k = 10)
{ // TODO étudiant : calculer le nombre de paramètres, comparer au dense pour 1,2,4,8,16,64
    return null;
}

Console.WriteLine($"Exercice 1 à compléter : {Budget(4)}");


Exercice 1 à compléter : 


## 3. Mini-tâche : adapter un classifieur gelé au « négatif photo »

Pour que la démonstration ait un enjeu, il faut un modèle de base **compétent sur son domaine d'entraînement et incompétent sur le domaine cible**. Fashion-MNIST s'y prête avec un décalage d'une ligne : l'**inversion** des pixels (`x -> 1 - x`), le négatif photo.

Le protocole :

1. entraîner un petit CNN (3 blocs conv + une tête linéaire, ~29 k paramètres) sur les images **inversées** ;
2. mesurer son exactitude sur le jeu de test inversé (son domaine) et sur le test normal (la cible) ;
3. geler tout le réseau, puis l'adapter au domaine normal **deux fois** : une avec LoRA `r = 4` sur deux couches, une en full fine-tuning des mêmes deux couches — mêmes epochs, même optimizer, même learning rate.

Un réseau composé uniquement de couches linéaires sans biais, ReLU et max-pool possède une homogénéité positive. Le CNN étudié possède des biais : cette propriété ne s'y applique pas directement. L’inversion choisie ici crée le décalage de domaine que les évaluations suivantes mesurent réellement.

In [5]:
// Lire les archives IDX, conserver le corpus complet et définir les lots sans remise.
var dossier = Environment.GetEnvironmentVariable("COURSIA_FASHION_MNIST") ?? throw new Exception("Définir COURSIA_FASHION_MNIST vers les archives Fashion-MNIST");
int Entier(Stream s)
{
    Span<byte> b = stackalloc byte[4];
    s.ReadExactly(b);
    return BinaryPrimitives.ReadInt32BigEndian(b);
}

(Tensor, Tensor) Charger(string prefix, int count)
{
    using var fi = File.OpenRead(Path.Combine(dossier, $"{prefix}-images-idx3-ubyte.gz"));
    using var im = new GZipStream(fi, CompressionMode.Decompress);
    using var fl = File.OpenRead(Path.Combine(dossier, $"{prefix}-labels-idx1-ubyte.gz"));
    using var la = new GZipStream(fl, CompressionMode.Decompress);
    if (Entier(im) != 2051 || Entier(im) != count || Entier(im) != 28 || Entier(im) != 28 || Entier(la) != 2049 || Entier(la) != count)
        throw new InvalidDataException("IDX inattendu");
    var pixels = new byte[count * 28 * 28];
    var labels = new byte[count];
    im.ReadExactly(pixels);
    la.ReadExactly(labels);
    return (torch.tensor(pixels).to_type(ScalarType.Float32).div_(255).reshape(count, 1, 28, 28), torch.tensor(labels).to_type(ScalarType.Int64));
}

var(images, labels) = Charger("train", 60000);
var(testImages, testLabels) = Charger("t10k", 10000);
Console.WriteLine($"Fashion-MNIST : {images.shape[0]} entraînement, {testImages.shape[0]} test");
public class SmallCNN : torch.nn.Module<Tensor, Tensor>
{
    public readonly Conv2d c1, c2, c3;
    public readonly Linear fc;
    public SmallCNN() : base("SmallCNN")
    {
        c1 = torch.nn.Conv2d(1, 16, 3, padding: 1);
        c2 = torch.nn.Conv2d(16, 32, 3, padding: 1);
        c3 = torch.nn.Conv2d(32, 64, 3, padding: 1);
        fc = torch.nn.Linear(576, 10);
        RegisterComponents();
    }

    public Tensor Prefix(Tensor x)
    {
        x = torch.nn.functional.max_pool2d(torch.nn.functional.relu(c1.forward(x)), 2);
        return torch.nn.functional.max_pool2d(torch.nn.functional.relu(c2.forward(x)), 2);
    }

    public override Tensor forward(Tensor x)
    {
        x = Prefix(x);
        return fc.forward(torch.nn.functional.max_pool2d(torch.nn.functional.relu(c3.forward(x)), 2).flatten(1));
    }
}

double Eval(Func<Tensor, Tensor> predict, bool inverse)
{
    using var noGrad = torch.no_grad();
    long good = 0;
    for (int i = 0; i < 10000; i += 512)
    {
        using var s = torch.NewDisposeScope();
        var x = testImages.narrow(0, i, Math.Min(512, 10000 - i));
        if (inverse)
            x = 1 - x;
        var y = testLabels.narrow(0, i, Math.Min(512, 10000 - i));
        good += predict(x).argmax(1).eq(y).sum().item<long>();
    }

    return good / 10000.0;
}

void Train(Func<Tensor, Tensor> predict, IEnumerable<Parameter> parameters, bool inverse, int epochs = 2)
{
    using var opt = torch.optim.Adam(parameters, lr: 1e-3);
    for (int ep = 1; ep <= epochs; ep++)
    {
        // Un tirage sans remise présente chaque image une fois par époque, y compris le dernier lot incomplet.
        using var perm = torch.randperm(60000);
        var timer = Stopwatch.StartNew();
        float last = 0;
        for (int i = 0; i < 60000; i += 256)
        {
            using var scope = torch.NewDisposeScope();
            var idx = perm.narrow(0, i, Math.Min(256, 60000 - i));
            var x = images.index_select(0, idx);
            if (inverse)
                x = 1 - x;
            var loss = torch.nn.functional.cross_entropy(predict(x), labels.index_select(0, idx));
            opt.zero_grad();
            loss.backward();
            opt.step();
            last = loss.item<float>();
        }

        Console.WriteLine($"époque {ep}/{epochs} : perte dernier mini-lot {last:F4}, {timer.Elapsed.TotalSeconds:F1} s, 60000 images");
    }
}

using (var m = new SmallCNN())
{
    var count = m.parameters().Sum(p => p.numel());
    Console.WriteLine($"paramètres CNN : {count}");
    if (count != 29066)
        throw new Exception("architecture incorrecte");
}


Fashion-MNIST : 60000 entraînement, 10000 test


paramètres CNN : 29066



warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy



### Entraînement du modèle de base (sur images inversées)

In [6]:
// Pré-entraîner deux époques sur le domaine inversé puis évaluer les deux domaines.
torch.manual_seed(42);
var baseModel = new SmallCNN();
Train(baseModel.forward, baseModel.parameters(), true);
var accInv = Eval(baseModel.forward, true);
var accBase = Eval(baseModel.forward, false);
Console.WriteLine($"Base, test inversé {accInv:F4}, test normal {accBase:F4}");


époque 1/2 : perte dernier mini-lot 0.5356, 19.0 s, 60000 images


époque 2/2 : perte dernier mini-lot 0.4993, 19.3 s, 60000 images


Base, test inversé 0.8380, test normal 0.0396


### Domaine inversé

Les deux exactitudes affichées mesurent le modèle entraîné sur négatifs puis évalué sur les domaines inversé et normal.

L’entraînement de base utilise toutes les images du jeu d’entraînement, transformées en négatif par `1 - x`. Le jeu de test est ensuite évalué deux fois : sur les négatifs, qui correspondent au domaine appris, puis sur les images normales, qui constituent la cible de l’adaptation. Les étiquettes ne changent pas : une chaussure reste une chaussure, seule la distribution des pixels change.

Ce contraste est un diagnostic du décalage de domaine, pas une mesure de généralisation à tous les décalages. Le compteur d’images et les pertes d’époque affichés permettent de vérifier le budget de calcul. Les durées proviennent de ce CPU et peuvent varier sous contention ; la comparaison pédagogique porte sur les mêmes images et les mêmes couches plutôt que sur un temps figé.

## 4. Deux adaptations du même réseau gelé : LoRA `r = 4` vs full

On gèle une copie du modèle de base, puis on construit deux adaptations :

- **LoRA** : deux adaptateurs — le dernier bloc convolutif et la tête linéaire, chacun en rang 4. Le reste du réseau (les deux premiers blocs) reste gelé.
- **full** : les mêmes deux couches, entièrement dégelées.

Même nombre d'epochs, même optimizer, même learning rate. Les deux conditions diffèrent par leur nombre de paramètres, leur sous-espace et leur paramétrisation : LoRA gèle les biais de base tandis que full les dégèle. La comparaison mesure l’effet conjoint de ces choix.

Pour la couche convolutive, la décomposition a une subtilité par rapport au cas linéaire : `A` est une convolution `in -> r` (noyau `k x k`) et `B` une convolution `1 x 1` `r -> out`. Le budget devient `r * (in * k² + out)`, et non `r * (in + out)` — le noyau 3 x 3 pèse neuf fois.

In [7]:
// Copier les poids initiaux et adapter exactement conv3 et la tête dans les deux conditions.
public class LoRAConv : torch.nn.Module<Tensor, Tensor>
{
    public readonly Conv2d Base, A, B;
    public readonly double Scaling;
    public LoRAConv(Conv2d b, int rank, double alpha) : base("LoRAConv")
    {
        Base = b;
        foreach (var p in b.parameters())
            p.requires_grad_(false);
        Scaling = alpha / rank;
        A = torch.nn.Conv2d(32, rank, 3, padding: 1, bias: false);
        B = torch.nn.Conv2d(rank, 64, 1, bias: false);
        using (var ng = torch.no_grad())
        {
            A.weight.uniform_(-1 / Math.Sqrt(32 * 9), 1 / Math.Sqrt(32 * 9));
            B.weight.zero_();
        }

        RegisterComponents();
    }

    public override Tensor forward(Tensor x) => Base.forward(x) + Scaling * B.forward(A.forward(x));
}

public class AdaptCNN : torch.nn.Module<Tensor, Tensor>
{
    public readonly SmallCNN Base;
    public readonly LoRAConv Conv;
    public readonly LoRALinear Head;
    public AdaptCNN(SmallCNN b) : base("AdaptCNN")
    {
        Base = b;
        foreach (var p in b.parameters())
            p.requires_grad_(false);
        Conv = new LoRAConv(b.c3, 4, 8);
        Head = new LoRALinear(b.fc, 4, 8);
        RegisterComponents();
    }

    public override Tensor forward(Tensor x)
    {
        x = Base.Prefix(x);
        x = torch.nn.functional.max_pool2d(torch.nn.functional.relu(Conv.forward(x)), 2).flatten(1);
        return Head.forward(x);
    }
}

// Les deux copies reçoivent les poids préentraînés pour isoler le choix du mode d’adaptation.
var frozenCopy = new SmallCNN();
frozenCopy.load_state_dict(baseModel.state_dict());
var loraModel = new AdaptCNN(frozenCopy);
// Les deux adaptateurs sont nuls à l'initialisation : vérifier le réseau entier sur les mêmes images.
using (var noGrad = torch.no_grad())
using (var scope = torch.NewDisposeScope())
{
    var sample = images.narrow(0, 0, 32);
    var copyError = (loraModel.forward(sample) - baseModel.forward(sample)).abs().max().item<float>();
    Console.WriteLine($"Identité initiale du réseau adapté : écart {copyError:E3}");
    if (copyError != 0)
        throw new Exception("la copie initiale diffère");
}

var fullModel = new SmallCNN();
fullModel.load_state_dict(baseModel.state_dict());
foreach (var p in fullModel.parameters())
    p.requires_grad_(false);
foreach (var p in fullModel.c3.parameters().Concat(fullModel.fc.parameters()))
    p.requires_grad_(true);
var loraParams = loraModel.parameters().Where(p => p.requires_grad).ToArray();
var fullParams = fullModel.parameters().Where(p => p.requires_grad).ToArray();
long nLora = loraParams.Sum(p => p.numel()), nFull = fullParams.Sum(p => p.numel());
Console.WriteLine($"LoRA entraînables {nLora}; full mêmes couches {nFull}");
if (nLora != 3752 || nFull != 24266)
    throw new Exception("budgets inattendus");


Identité initiale du réseau adapté : écart 0.000E+000


LoRA entraînables 3752; full mêmes couches 24266


Les deux adaptations sont entraînées et évaluées sur les mêmes domaines.

In [8]:
// Entraîner les deux adaptations sur les mêmes images et comparer leurs budgets.
Console.WriteLine("LoRA, domaine normal :");
torch.manual_seed(42);
Train(loraModel.forward, loraParams, false);
var accLora = Eval(loraModel.forward, false);
Console.WriteLine("Full, mêmes couches :");
torch.manual_seed(42);
Train(fullModel.forward, fullParams, false);
var accFull = Eval(fullModel.forward, false);
Console.WriteLine($"base inversé {accInv:F4}, base normal {accBase:F4}, LoRA {accLora:F4} ({nLora} paramètres), full {accFull:F4} ({nFull} paramètres)");


LoRA, domaine normal :


époque 1/2 : perte dernier mini-lot 1.0467, 14.5 s, 60000 images


époque 2/2 : perte dernier mini-lot 0.7218, 10.0 s, 60000 images


Full, mêmes couches :


époque 1/2 : perte dernier mini-lot 0.4247, 9.5 s, 60000 images


époque 2/2 : perte dernier mini-lot 0.3501, 8.4 s, 60000 images


base inversé 0.8380, base normal 0.0396, LoRA 0.7020 (3752 paramètres), full 0.8368 (24266 paramètres)


### Comparaison des adaptations

Les deux méthodes partent des mêmes poids et adaptent les mêmes couches. Comparez les exactitudes et les budgets réellement affichés ; les résultats dépendent de l’entraînement CPU.

La branche LoRA ajoute un adaptateur convolutif sur le troisième bloc et un adaptateur linéaire sur la tête. Pour la convolution, A est un noyau `3 × 3` de rang 4 et B une projection `1 × 1` : leur budget est `4 × (32 × 3 × 3) + 64 × 4 = 1 408`. La tête ajoute `4 × (576 + 10) = 2 344` paramètres, soit `3 752` au total.

La branche full dégèle les paramètres de ces mêmes deux couches, soit `24 266` paramètres ; les deux premiers blocs restent gelés. Les deux branches démarrent d’une copie des mêmes poids pré-entraînés, utilisent deux époques sur les images normales, Adam et le même taux d’apprentissage. Le rapport de budgets est donc structurel ; l’écart d’exactitude est une mesure de ce run. Des initialisations d’adaptateur et des ordres de lots peuvent faire varier les scores même à architecture et graine données.

### Exercice 2 — Découpler `alpha` du rang

Compléter `CompareAlpha` sur une même petite tâche de régression, par exemple une cible sinusoïdale bruitée. Entraîner deux `LoRALinear` de rang 4 avec `alpha = 4` puis `alpha = 16`, chaque fois pendant 200 pas d’Adam à `1e-2`. Repartir des mêmes poids de base, de la même graine et des mêmes lots.

**À comparer.** Relever la perte finale et la norme de la correction effective dans les deux cas. Le multiplicateur passe de 1 à 4 pour des facteurs A et B fixés. À l’issue de l’optimisation, les facteurs peuvent différer : que mesurez-vous au même pas d’apprentissage ?


In [9]:
(double, double)? CompareAlpha(int d = 64, int k = 8, int steps = 200)
{ // TODO étudiant : comparer alpha 4 et 16, rang 4, mêmes graines et Adam 1e-2
    return null;
}

Console.WriteLine($"Exercice 2 à compléter : {CompareAlpha()}");


Exercice 2 à compléter : 


## 5. Fusion du poids et précision flottante

Après entraînement, former `W_fusionné = W + (alpha/r) * B @ A` et conserver le biais. Une couche linéaire dense TorchSharp exécute alors la prédiction sans les deux produits bas rang du chemin d’adaptation.

Les deux expressions sont égales en arithmétique exacte. En IEEE 754, réordonner les produits et les additions change les points d’arrondi. La cellule compare les formes fusionnée et non fusionnée sur les mêmes entrées en simple puis en double précision.


In [10]:
// Fusionner les poids et comparer la dérive en simple puis double précision.
var loraFc = loraModel.Head;
var xProbe = torch.randn(new long[] { 128, 576 });
var merged = loraFc.MergedWeight();
var yMerged = torch.nn.functional.linear(xProbe, merged, loraFc.Base.bias);
var drift32 = (yMerged - loraFc.forward(xProbe)).abs().max().item<float>();
var x64 = torch.randn(new long[] { 128, 576 }, dtype: ScalarType.Float64);
var W = loraFc.Base.weight.to_type(ScalarType.Float64);
var bias = loraFc.Base.bias.to_type(ScalarType.Float64);
var A = loraFc.A.to_type(ScalarType.Float64);
var B = loraFc.B.to_type(ScalarType.Float64);
var y1 = torch.nn.functional.linear(x64, W, bias) + loraFc.Scaling * torch.matmul(torch.matmul(x64, A.transpose(0, 1)), B.transpose(0, 1));
var y2 = torch.nn.functional.linear(x64, W + loraFc.Scaling * torch.matmul(B, A), bias);
var drift64 = (y2 - y1).abs().max().item<double>();
var dn = (loraFc.Scaling * torch.matmul(loraFc.B, loraFc.A)).norm().item<float>();
var wn = loraFc.Base.weight.norm().item<float>();
Console.WriteLine($"dérive float32 {drift32:E3}, float64 {drift64:E3}; norme delta {dn:F4}, norme W {wn:F4}, rapport {dn / wn:P1}");


dérive float32 2.146E-006, float64 3.553E-015; norme delta 2.7278, norme W 3.3133, rapport 82.3 %


### Fusion et précision

La fusion change l’ordre des opérations flottantes. La cellule affiche les dérives observées en simple et double précision et la norme du delta relativement au poids de base.

À l’inférence, la tête adaptée peut être réécrite avec un poids dense `W_fusionné = W + (alpha/r) B A` et le même biais. Cette fusion élimine les deux produits bas rang du chemin de prédiction, mais elle ne change pas la fonction mathématique visée. La sortie de la cellule compare les deux chemins sur des entrées aléatoires, après un vrai entraînement de l’adaptateur.

La différence observée n’est pas nécessairement nulle : calculer `x Wᵀ + scaling × ((x Aᵀ) Bᵀ)` ou calculer `x (W + scaling × B A)ᵀ` regroupe les opérations et les arrondis différemment. Le contrôle en float64 reprend les poids entraînés convertis en double et refait les deux chemins ; il aide à distinguer une erreur d’implémentation d’un écart lié à la précision. La norme du delta rapporte l’ampleur de la modification à celle du poids de base, sans la confondre avec l’exactitude.

### Exercice 3 : ré-écrire la fusion et borner sa dérive

`MergedWeight()` existe déjà ; écrivez sa contre-vérification : reconstruisez la couche fusionnée, comparez-la à la couche adaptée sur un lot d'entrées, et bornez la dérive en fonction de la précision.

In [11]:
double? DeriveFusion(LoRALinear couche, int n = 256)
{ // TODO étudiant : fusionner poids et biais, comparer en float32 puis float64
    return null;
}

Console.WriteLine($"Exercice 3 à compléter : {DeriveFusion(loraModel.Head)}");


Exercice 3 à compléter : 


## Résumé

| notion | ce qu'il faut retenir |
|---|---|
| la décomposition | `W' = W + (alpha/r) * B @ A`, budget `r (d + k)` au lieu de `d x k` (conv : `r (in k² + out)`) |
| l'initialisation | `A ~ N`, `B = 0` ⇒ `W' = W` **exactement** au pas 0 — vérifiable en un test |
| le gel | `requires_grad_(False)` évite le gradient de W, sans empêcher celui de l’entrée lorsqu’il est nécessaire |
| l'entraînement | seuls `A`, `B` bougent ; l'adaptateur se stocke en `r (d + k)` valeurs |
| la mesure | rang 4 sur deux couches, 3 752 valeurs apprises contre 24 266 pour l’adaptation complète des mêmes couches |
| la fusion | même fonction en arithmétique exacte, avec une dérive flottante à mesurer |

**Étape suivante** : le même mécanisme à l'échelle industrielle — [FT-01](FT-01-Introduction-FineTuning-Python.ipynb) (LoRA `peft` sur GPT-2), [FT-02](FT-02-QLoRA-Quantization-Python.ipynb) (QLoRA : base 4-bit + adaptateurs), [FT-06](FT-06-Vision-Language-LoRA-Python.ipynb) (vision-langage). Vous y reconnaîtrez chaque pièce que vous venez d'écrire à la main.